## Goal

> ⚠️ **Research on public allocation data as of 2026-W44.** Not investment advice; results describe the methods, not the funds.

> 🇸🇪 **Categories and cluster names are in Swedish**, as delivered by the source. Common words: *Branschfond* = sector fund, *Sverige* = Sweden, *Norden* = Nordics, *Små-/medelstora bolag* = small/mid caps, *Mix bolag* = mixed, *Råvaror* = commodities, *Läkemedel* = pharma.

Find **similar funds** for search. Existing categories (e.g. "Branschfond, Ny teknik") are too generic, so funds are compared on their **country and sector percentages** instead. Each fund's percentages are turned into a list of numbers, so funds with similar holdings end up **close together**. The three methods differ mainly in how much **small holdings** count:

- **Cosine**: looks at the overall mix. Big holdings decide; small ones barely matter.
- **Aitchison**: compares holdings with each other, so a small holding can matter as much as a big one.
- **Hellinger 60/40**: a middle way. Small holdings count a little more than in Cosine, and countries count a bit more than sectors.

To find similar funds, every other fund is ranked by how close it is, closest first (nearest neighbours).

## How clusters are made and named

1. **Simplify (UMAP)**: each fund is described by about 100 numbers, too many to group well. UMAP boils them down to 10, keeping funds that were alike still close together.
2. **Group (HDBSCAN)**: HDBSCAN looks for places where many funds sit close together, at least 5. Each such group is a cluster. It decides by itself how many clusters there are. Funds that don't fit any group are "Not in a cluster".
3. **Name**: only now do we look at the categories. Each cluster is named after its most common category. If no category has at least half of the funds, the name shows the top two: "Mixed · A / B". When two clusters get the same name, they're labelled " · A", " · B".
4. **Compare**: for each fund, we check whether its own category **agrees** with the cluster's name, is **neighbouring** (same region, e.g. Sverige vs Norden), or is from a **different family**.

In [ ]:
# Re-import edited fund_clustering modules automatically before each cell runs.
%load_ext autoreload
%autoreload 2

In [ ]:
# ISO week to analyse; its CSVs must be in ../data. The text and summary
# below describe 2026-W44.
WEEK = "2026-W44"

In [ ]:
# Load the WEEK export: allocations and metadata, equity funds only.
import sys
from pathlib import Path

from fund_clustering.load import load_week

# Should point into FundClustering\.venv; anything else means the wrong kernel.
print(sys.executable)

DATA_DIR = Path("../data")  # CSV exports, example weeks committed

data = load_week(DATA_DIR, WEEK)

# Week, (funds, columns), country column count, sector column count
WEEK, data.funds.shape, len(data.country_columns), len(data.sector_columns)


## Example funds

Used in the neighbours cell of each method to check which cluster a fund lands in and which funds are most similar.

| ISIN | Name | Category | Why |
|------|------|----------|-----|
| `LU0256331488` | Schroder ISF Global Energy A Acc USD | Branschfond, Energi | Category search misses its peers: natural resources funds hold a lot of energy too, but sit in "Branschfond, Råvaror". A good method should find them by holdings. |
| `NO0013528349` | DNB European Defence S | Branschfond, Industrimaterial | Harder case: the category is generic and the countries look like an ordinary European fund. Only the sector mix (industrials plus tech) shows the defence theme. Can a method still find the other defence funds? |
| `LU0069152568` | UBS (Lux) EF Biotech (USD) P-acc | Branschfond, Bioteknik | The right answer judged "wrong": its neighbours are biotech, then healthcare funds. Cosine gives biotech its own cluster, but Aitchison and Hellinger put it in "Branschfond, Läkemedel", where it counts as a different family. Shows the verdict rule is too strict for related sectors. |
| `LU0256846568` | Pictet-Security R USD | Global, Flexibel | Vague category: the holdings are US tech and growth, and its neighbours show that better than the category does. |
| `SE0015837778` | Go Blockchain Fund A | Branschfond, Ny teknik | Known limit: the blockchain theme isn't visible in country and sector data, so it looks like a US index fund. No method can fix this. |

## Cosine

All country and sector percentages as one vector per fund, scaled to length 1, so funds are compared by the **mix** of their holdings, not size. Short positions are used as reported. No weights between countries and sectors. The default method on the map.

In [ ]:
# Cosine features: one vector per fund (countries and sectors), scaled to length 1
# so funds are compared by allocation mix, not size.
import numpy as np

from fund_clustering.methods.cosine import cosine_features

X = cosine_features(data)

# Sanity check: every row should have length 1 (tiny rounding is fine).
lengths = np.linalg.norm(X, axis=1)

X.shape, lengths.min(), lengths.max()


In [ ]:
# Clustering: 10D UMAP + HDBSCAN gives a cluster id per fund (-1 = not in a
# cluster); a separate 2D UMAP gives the map position. ~20 s on the first run.
from fund_clustering.cluster import cluster_funds

result = cluster_funds(X)

# Working table for the cells below: fund data plus cluster and map position.
funds = data.funds.assign(
    cluster=result.labels, x=result.xy[:, 0], y=result.xy[:, 1]
)

# (number of clusters, funds not in a cluster)
result.cluster_count, result.not_in_cluster_count


In [ ]:
# Naming: each cluster is named after its most common category ("Mixed · A / B"
# under 50% purity); each fund gets a verdict comparing its category to that.
from fund_clustering.naming import describe_clusters, fund_verdicts

clusters = describe_clusters(funds["category"], result.labels)
funds["verdict"] = fund_verdicts(funds["category"], result.labels, clusters)
funds["cluster_name"] = funds["cluster"].map(clusters["name"]).fillna("Not in a cluster")

# agree / neighbouring (same regional family) / different (the other family) / none
display(funds["verdict"].value_counts())

# Largest clusters first; purity = share of the leading category.
clusters[["name", "size", "purity"]].sort_values("size", ascending=False)


In [ ]:
# Neighbours: the 20 most similar funds per fund, measured on the real allocation
# vectors (not the 2D map). Isolated = 5% of funds farthest from their 5th neighbour.
import pandas as pd

from fund_clustering.neighbours import isolated_flags, nearest_neighbours

neighbours = nearest_neighbours(X)
funds["isolated"] = isolated_flags(neighbours)


def similar_funds(isin: str, count: int = 10) -> pd.DataFrame:
    """Most similar funds to ``isin``, closest first (distance 0 = identical mix)."""
    row = funds.index[funds["isin"] == isin][0]
    similar = funds.iloc[neighbours.indices[row, :count]]
    return similar[["isin", "name", "category", "cluster_name"]].assign(
        distance=neighbours.distances[row, :count].round(3)
    )


# Example funds (see the table at the top); try any ISIN here.
# Schroder ISF Global Energy A Acc USD, "Branschfond, Energi": peers sit in Råvaror
ENERGY_ISIN = "LU0256331488"
# DNB European Defence S, "Branschfond, Industrimaterial": theme only in the sectors
DEFENCE_ISIN = "NO0013528349"
# UBS (Lux) EF Biotech (USD) P-acc, "Branschfond, Bioteknik": right peers, strict verdict
BIOTECH_ISIN = "LU0069152568"
# Pictet-Security R USD, "Global, Flexibel": vague category, US tech holdings
SECURITY_ISIN = "LU0256846568"
# Go Blockchain Fund A, "Branschfond, Ny teknik": theme not visible in the data
BLOCKCHAIN_ISIN = "SE0015837778"

for isin in (ENERGY_ISIN, DEFENCE_ISIN, BIOTECH_ISIN, SECURITY_ISIN, BLOCKCHAIN_ISIN):
    selected = funds.loc[funds["isin"] == isin].iloc[0]
    print(selected["name"], "|", selected["category"], "|", selected["cluster_name"])
    display(similar_funds(isin))


In [ ]:
# Territory: map background per cluster. A ~220-cell grid; each cell takes the
# distance-weighted vote of its 7 nearest clustered funds within a radius,
# then each cluster's cells are merged into polygons (drawn in the map below).
import shapely

from fund_clustering.territory import territory_grid, territory_polygons

grid = territory_grid(result.xy, result.labels)
territories = territory_polygons(grid)

# (grid rows x cols, territories, polygon parts, total vertices to export)
parts = sum(len(getattr(shape, "geoms", [shape])) for shape in territories.values())
vertices = sum(shapely.get_num_coordinates(shape) for shape in territories.values())
grid.labels.shape, len(territories), parts, vertices


In [ ]:
# Map: one dot per fund, coloured by cluster, over its cluster's territory.
# Scroll to zoom, drag to pan, hover for details. Only local closeness is
# meaningful, not distances between far-apart groups.
import plotly.express as px
import plotly.graph_objects as go

# Outline colour by verdict: amber = neighbouring, red = different family.
VERDICT_OUTLINE = {
    "agree": "rgba(0,0,0,0)",
    "neighbouring": "#c58300",
    "different": "#d0353a",
    "none": "rgba(0,0,0,0)",
}

# One colour per cluster, shared by dots and territory; grey = not in a cluster.
PALETTE = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
cluster_colours = {
    str(cluster_id): PALETTE[i % len(PALETTE)]
    for i, cluster_id in enumerate(sorted(clusters.index))
}
cluster_colours["-1"] = "#9aa0a6"


def with_alpha(hex_colour: str, alpha: float) -> str:
    r, g, b = (int(hex_colour[i : i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alpha})"


fig = px.scatter(
    funds.assign(cluster=funds["cluster"].astype(str)),  # str -> discrete colours
    x="x",
    y="y",
    color="cluster",
    color_discrete_map=cluster_colours,
    hover_name="name",
    hover_data=[
        "isin", "category", "cluster_name", "verdict", "isolated", "company_name"
    ],
    height=750,
)
# plotly makes one trace per cluster; give each trace's dots their outlines.
for trace in fig.data:
    rows = funds["cluster"].astype(str) == trace.name
    trace.marker.line = dict(
        color=funds.loc[rows, "verdict"].map(VERDICT_OUTLINE).tolist(), width=2
    )

# Territory polygons (outer edges only), added last, then moved behind the dots.
dot_traces = len(fig.data)
for cluster_id, shape in territories.items():
    colour = cluster_colours[str(cluster_id)]
    for polygon in getattr(shape, "geoms", [shape]):
        x, y = polygon.exterior.xy
        fig.add_trace(
            go.Scatter(
                x=list(x),
                y=list(y),
                fill="toself",
                fillcolor=with_alpha(colour, 0.22),
                line=dict(color=with_alpha(colour, 0.6), width=0.5),
                mode="lines",
                hoverinfo="skip",
                showlegend=False,
            )
        )
fig.data = fig.data[dot_traces:] + fig.data[:dot_traces]

fig.update_layout(
    title=f"Cosine: {result.cluster_count} clusters, "
    f"{result.not_in_cluster_count} not in a cluster"
)
fig.update_yaxes(scaleanchor="x")  # same scale on both axes, no stretching
fig.show()


## Aitchison

Centred log-ratio of the country shares and, separately, of the sector shares: `log(p + 0.005)` minus its mean. Distance compares the **ratios** between holdings, so small positions count as much as large ones. Short positions read as 0. Distances are on a different scale than Cosine (closest match ≈ 3 instead of ≈ 0.1), so compare the order, not the numbers.

In [ ]:
# Aitchison features: centred log-ratio of the country shares and, separately, of
# the sector shares, so funds are compared by the ratios between holdings.
import numpy as np

from fund_clustering.methods.aitchison import aitchison_features

X = aitchison_features(data)

# Sanity check: the country part and the sector part each sum to 0 per fund
# (tiny rounding is fine).
countries = len(data.country_columns)

X.shape, np.abs(X[:, :countries].sum(axis=1)).max(), np.abs(X[:, countries:].sum(axis=1)).max()


In [ ]:
# Clustering: 10D UMAP + HDBSCAN gives a cluster id per fund (-1 = not in a
# cluster); a separate 2D UMAP gives the map position. ~20 s on the first run.
from fund_clustering.cluster import cluster_funds

result = cluster_funds(X)

# Working table for the cells below: fund data plus cluster and map position.
funds = data.funds.assign(
    cluster=result.labels, x=result.xy[:, 0], y=result.xy[:, 1]
)

# (number of clusters, funds not in a cluster)
result.cluster_count, result.not_in_cluster_count


In [ ]:
# Naming: each cluster is named after its most common category ("Mixed · A / B"
# under 50% purity); each fund gets a verdict comparing its category to that.
from fund_clustering.naming import describe_clusters, fund_verdicts

clusters = describe_clusters(funds["category"], result.labels)
funds["verdict"] = fund_verdicts(funds["category"], result.labels, clusters)
funds["cluster_name"] = funds["cluster"].map(clusters["name"]).fillna("Not in a cluster")

# agree / neighbouring (same regional family) / different (the other family) / none
display(funds["verdict"].value_counts())

# Largest clusters first; purity = share of the leading category.
clusters[["name", "size", "purity"]].sort_values("size", ascending=False)


In [ ]:
# Neighbours: the 20 most similar funds per fund, measured on the real allocation
# vectors (not the 2D map). Isolated = 5% of funds farthest from their 5th neighbour.
import pandas as pd

from fund_clustering.neighbours import isolated_flags, nearest_neighbours

neighbours = nearest_neighbours(X)
funds["isolated"] = isolated_flags(neighbours)


def similar_funds(isin: str, count: int = 10) -> pd.DataFrame:
    """Most similar funds to ``isin``, closest first (distance 0 = identical mix)."""
    row = funds.index[funds["isin"] == isin][0]
    similar = funds.iloc[neighbours.indices[row, :count]]
    return similar[["isin", "name", "category", "cluster_name"]].assign(
        distance=neighbours.distances[row, :count].round(3)
    )


# Example funds (see the table at the top); try any ISIN here.
# Schroder ISF Global Energy A Acc USD, "Branschfond, Energi": peers sit in Råvaror
ENERGY_ISIN = "LU0256331488"
# DNB European Defence S, "Branschfond, Industrimaterial": theme only in the sectors
DEFENCE_ISIN = "NO0013528349"
# UBS (Lux) EF Biotech (USD) P-acc, "Branschfond, Bioteknik": right peers, strict verdict
BIOTECH_ISIN = "LU0069152568"
# Pictet-Security R USD, "Global, Flexibel": vague category, US tech holdings
SECURITY_ISIN = "LU0256846568"
# Go Blockchain Fund A, "Branschfond, Ny teknik": theme not visible in the data
BLOCKCHAIN_ISIN = "SE0015837778"

for isin in (ENERGY_ISIN, DEFENCE_ISIN, BIOTECH_ISIN, SECURITY_ISIN, BLOCKCHAIN_ISIN):
    selected = funds.loc[funds["isin"] == isin].iloc[0]
    print(selected["name"], "|", selected["category"], "|", selected["cluster_name"])
    display(similar_funds(isin))


In [ ]:
# Territory: map background per cluster. A ~220-cell grid; each cell takes the
# distance-weighted vote of its 7 nearest clustered funds within a radius,
# then each cluster's cells are merged into polygons (drawn in the map below).
import shapely

from fund_clustering.territory import territory_grid, territory_polygons

grid = territory_grid(result.xy, result.labels)
territories = territory_polygons(grid)

# (grid rows x cols, territories, polygon parts, total vertices to export)
parts = sum(len(getattr(shape, "geoms", [shape])) for shape in territories.values())
vertices = sum(shapely.get_num_coordinates(shape) for shape in territories.values())
grid.labels.shape, len(territories), parts, vertices


In [ ]:
# Map: one dot per fund, coloured by cluster, over its cluster's territory.
# Scroll to zoom, drag to pan, hover for details. Only local closeness is
# meaningful, not distances between far-apart groups.
import plotly.express as px
import plotly.graph_objects as go

# Outline colour by verdict: amber = neighbouring, red = different family.
VERDICT_OUTLINE = {
    "agree": "rgba(0,0,0,0)",
    "neighbouring": "#c58300",
    "different": "#d0353a",
    "none": "rgba(0,0,0,0)",
}

# One colour per cluster, shared by dots and territory; grey = not in a cluster.
PALETTE = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
cluster_colours = {
    str(cluster_id): PALETTE[i % len(PALETTE)]
    for i, cluster_id in enumerate(sorted(clusters.index))
}
cluster_colours["-1"] = "#9aa0a6"


def with_alpha(hex_colour: str, alpha: float) -> str:
    r, g, b = (int(hex_colour[i : i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alpha})"


fig = px.scatter(
    funds.assign(cluster=funds["cluster"].astype(str)),  # str -> discrete colours
    x="x",
    y="y",
    color="cluster",
    color_discrete_map=cluster_colours,
    hover_name="name",
    hover_data=[
        "isin", "category", "cluster_name", "verdict", "isolated", "company_name"
    ],
    height=750,
)
# plotly makes one trace per cluster; give each trace's dots their outlines.
for trace in fig.data:
    rows = funds["cluster"].astype(str) == trace.name
    trace.marker.line = dict(
        color=funds.loc[rows, "verdict"].map(VERDICT_OUTLINE).tolist(), width=2
    )

# Territory polygons (outer edges only), added last, then moved behind the dots.
dot_traces = len(fig.data)
for cluster_id, shape in territories.items():
    colour = cluster_colours[str(cluster_id)]
    for polygon in getattr(shape, "geoms", [shape]):
        x, y = polygon.exterior.xy
        fig.add_trace(
            go.Scatter(
                x=list(x),
                y=list(y),
                fill="toself",
                fillcolor=with_alpha(colour, 0.22),
                line=dict(color=with_alpha(colour, 0.6), width=0.5),
                mode="lines",
                hoverinfo="skip",
                showlegend=False,
            )
        )
fig.data = fig.data[dot_traces:] + fig.data[:dot_traces]

fig.update_layout(
    title=f"Aitchison: {result.cluster_count} clusters, "
    f"{result.not_in_cluster_count} not in a cluster"
)
fig.update_yaxes(scaleanchor="x")  # same scale on both axes, no stretching
fig.show()


## Hellinger 60/40

Square root of the country shares and, separately, of the sector shares, each renormalised to 100% first. The square root lifts small positions a little, without the log's blow-up in Aitchison. Countries weigh 60% and sectors 40% of the squared distance. Short positions read as 0. Distances run from 0 to √2, closer to the Cosine scale than to Aitchison.

In [ ]:
# Hellinger features: square root of the country shares and, separately, of the
# sector shares (each renormalised to 100%), weighted 60/40 between the two.
import numpy as np

from fund_clustering.methods.hellinger import hellinger_features

X = hellinger_features(data)

# Sanity check: per fund, the squared country part sums to 0.6 and the squared
# sector part to 0.4 (0 for a part with no long positions).
countries = len(data.country_columns)
country_sums = (X[:, :countries] ** 2).sum(axis=1).round(9)
sector_sums = (X[:, countries:] ** 2).sum(axis=1).round(9)

X.shape, np.unique(country_sums), np.unique(sector_sums)


In [ ]:
# Clustering: 10D UMAP + HDBSCAN gives a cluster id per fund (-1 = not in a
# cluster); a separate 2D UMAP gives the map position. ~20 s on the first run.
from fund_clustering.cluster import cluster_funds

result = cluster_funds(X)

# Working table for the cells below: fund data plus cluster and map position.
funds = data.funds.assign(
    cluster=result.labels, x=result.xy[:, 0], y=result.xy[:, 1]
)

# (number of clusters, funds not in a cluster)
result.cluster_count, result.not_in_cluster_count


In [ ]:
# Naming: each cluster is named after its most common category ("Mixed · A / B"
# under 50% purity); each fund gets a verdict comparing its category to that.
from fund_clustering.naming import describe_clusters, fund_verdicts

clusters = describe_clusters(funds["category"], result.labels)
funds["verdict"] = fund_verdicts(funds["category"], result.labels, clusters)
funds["cluster_name"] = funds["cluster"].map(clusters["name"]).fillna("Not in a cluster")

# agree / neighbouring (same regional family) / different (the other family) / none
display(funds["verdict"].value_counts())

# Largest clusters first; purity = share of the leading category.
clusters[["name", "size", "purity"]].sort_values("size", ascending=False)


In [ ]:
# Neighbours: the 20 most similar funds per fund, measured on the real allocation
# vectors (not the 2D map). Isolated = 5% of funds farthest from their 5th neighbour.
import pandas as pd

from fund_clustering.neighbours import isolated_flags, nearest_neighbours

neighbours = nearest_neighbours(X)
funds["isolated"] = isolated_flags(neighbours)


def similar_funds(isin: str, count: int = 10) -> pd.DataFrame:
    """Most similar funds to ``isin``, closest first (distance 0 = identical mix)."""
    row = funds.index[funds["isin"] == isin][0]
    similar = funds.iloc[neighbours.indices[row, :count]]
    return similar[["isin", "name", "category", "cluster_name"]].assign(
        distance=neighbours.distances[row, :count].round(3)
    )


# Example funds (see the table at the top); try any ISIN here.
# Schroder ISF Global Energy A Acc USD, "Branschfond, Energi": peers sit in Råvaror
ENERGY_ISIN = "LU0256331488"
# DNB European Defence S, "Branschfond, Industrimaterial": theme only in the sectors
DEFENCE_ISIN = "NO0013528349"
# UBS (Lux) EF Biotech (USD) P-acc, "Branschfond, Bioteknik": right peers, strict verdict
BIOTECH_ISIN = "LU0069152568"
# Pictet-Security R USD, "Global, Flexibel": vague category, US tech holdings
SECURITY_ISIN = "LU0256846568"
# Go Blockchain Fund A, "Branschfond, Ny teknik": theme not visible in the data
BLOCKCHAIN_ISIN = "SE0015837778"

for isin in (ENERGY_ISIN, DEFENCE_ISIN, BIOTECH_ISIN, SECURITY_ISIN, BLOCKCHAIN_ISIN):
    selected = funds.loc[funds["isin"] == isin].iloc[0]
    print(selected["name"], "|", selected["category"], "|", selected["cluster_name"])
    display(similar_funds(isin))


In [ ]:
# Territory: map background per cluster. A ~220-cell grid; each cell takes the
# distance-weighted vote of its 7 nearest clustered funds within a radius,
# then each cluster's cells are merged into polygons (drawn in the map below).
import shapely

from fund_clustering.territory import territory_grid, territory_polygons

grid = territory_grid(result.xy, result.labels)
territories = territory_polygons(grid)

# (grid rows x cols, territories, polygon parts, total vertices to export)
parts = sum(len(getattr(shape, "geoms", [shape])) for shape in territories.values())
vertices = sum(shapely.get_num_coordinates(shape) for shape in territories.values())
grid.labels.shape, len(territories), parts, vertices


In [ ]:
# Map: one dot per fund, coloured by cluster, over its cluster's territory.
# Scroll to zoom, drag to pan, hover for details. Only local closeness is
# meaningful, not distances between far-apart groups.
import plotly.express as px
import plotly.graph_objects as go

# Outline colour by verdict: amber = neighbouring, red = different family.
VERDICT_OUTLINE = {
    "agree": "rgba(0,0,0,0)",
    "neighbouring": "#c58300",
    "different": "#d0353a",
    "none": "rgba(0,0,0,0)",
}

# One colour per cluster, shared by dots and territory; grey = not in a cluster.
PALETTE = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
cluster_colours = {
    str(cluster_id): PALETTE[i % len(PALETTE)]
    for i, cluster_id in enumerate(sorted(clusters.index))
}
cluster_colours["-1"] = "#9aa0a6"


def with_alpha(hex_colour: str, alpha: float) -> str:
    r, g, b = (int(hex_colour[i : i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alpha})"


fig = px.scatter(
    funds.assign(cluster=funds["cluster"].astype(str)),  # str -> discrete colours
    x="x",
    y="y",
    color="cluster",
    color_discrete_map=cluster_colours,
    hover_name="name",
    hover_data=[
        "isin", "category", "cluster_name", "verdict", "isolated", "company_name"
    ],
    height=750,
)
# plotly makes one trace per cluster; give each trace's dots their outlines.
for trace in fig.data:
    rows = funds["cluster"].astype(str) == trace.name
    trace.marker.line = dict(
        color=funds.loc[rows, "verdict"].map(VERDICT_OUTLINE).tolist(), width=2
    )

# Territory polygons (outer edges only), added last, then moved behind the dots.
dot_traces = len(fig.data)
for cluster_id, shape in territories.items():
    colour = cluster_colours[str(cluster_id)]
    for polygon in getattr(shape, "geoms", [shape]):
        x, y = polygon.exterior.xy
        fig.add_trace(
            go.Scatter(
                x=list(x),
                y=list(y),
                fill="toself",
                fillcolor=with_alpha(colour, 0.22),
                line=dict(color=with_alpha(colour, 0.6), width=0.5),
                mode="lines",
                hoverinfo="skip",
                showlegend=False,
            )
        )
fig.data = fig.data[dot_traces:] + fig.data[:dot_traces]

fig.update_layout(
    title=f"Hellinger 60/40: {result.cluster_count} clusters, "
    f"{result.not_in_cluster_count} not in a cluster"
)
fig.update_yaxes(scaleanchor="x")  # same scale on both axes, no stretching
fig.show()


## Comparison: defence funds

DNB European Defence S lands in a broad Europe cluster under every method, so does its neighbour list do better? The table shows where the four other defence funds rank among its 20 nearest neighbours (empty = not in the top 20), for each method and for Hellinger with more weight on sectors. The extra weightings live only in this notebook.

In [ ]:
# Defence funds: rank of the other four in DNB European Defence S's top 20, per
# method, plus Hellinger with more weight on sectors (tried here only).
import numpy as np
import pandas as pd

from fund_clustering.methods import METHODS, allocation_matrix
from fund_clustering.methods.hellinger import square_root_shares
from fund_clustering.neighbours import nearest_neighbours

# The other defence funds, all "Branschfond, Industrimaterial".
OTHER_DEFENCE_ISINS = [
    "SE0025491764",  # SEB European Defence & Security A (SEK), Europe
    "DK0064195267",  # Danske Invest Defence & Security Acc SEK, USA ~60%
    "LU1822851538",  # Finserve Global Defence & Sec I SEK R, USA, Korea, Japan
    "SE0025667207",  # Swedbank Robur Security and Defence A, USA ~60%
]


def hellinger_weighted(country_weight: float) -> np.ndarray:
    """Hellinger features with a custom country/sector split."""
    matrix = allocation_matrix(data)
    countries = len(data.country_columns)
    return np.hstack([
        np.sqrt(country_weight) * square_root_shares(matrix[:, :countries]),
        np.sqrt(1 - country_weight) * square_root_shares(matrix[:, countries:]),
    ])


candidates = {method.name: method.features(data) for method in METHODS.values()}
for country_weight in (0.4, 0.3, 0.2, 0.0):
    label = f"Hellinger {country_weight * 100:.0f}/{(1 - country_weight) * 100:.0f}"
    candidates[label] = hellinger_weighted(country_weight)
neighbour_indices = {
    label: nearest_neighbours(X).indices for label, X in candidates.items()
}

isins = data.funds["isin"].to_numpy()
short_names = data.funds.set_index("isin")["name"].str[:24]


def peer_ranks(isin: str, peer_isins: list[str]) -> pd.DataFrame:
    """Rank (1 = closest) of each peer in ``isin``'s top 20, per candidate.

    Empty = not in the top 20.
    """
    row = np.flatnonzero(isins == isin)[0]
    ranks = {}
    for label, indices in neighbour_indices.items():
        top = list(isins[indices[row]])
        ranks[label] = [top.index(p) + 1 if p in top else None for p in peer_isins]
    return pd.DataFrame(ranks, index=short_names[peer_isins]).T.astype("Int64")


peer_ranks(DEFENCE_ISIN, OTHER_DEFENCE_ISINS)


## Comparison: energy funds

Schroder ISF Global Energy (energy 94%) should find the other oil & gas fund first, then the natural resources funds, which hold 32–55% energy. Clean-energy funds ("Branschfond, Ny energi") are left out on purpose: their holdings are mostly industrials and tech, a different theme. Same table as for defence: rank among the 20 nearest neighbours, empty = not in the top 20.

In [ ]:
# Energy: rank of the other oil & gas fund and the natural resources funds in
# Schroder ISF Global Energy's top 20 (same candidates as the defence cell).
ENERGY_PEER_ISINS = [
    "LU0122376428",  # BGF World Energy A2, "Branschfond, Energi", energy 97%
    "LU0208853274",  # JPM Global Natural Resources A acc EUR, energy 55%
    "LU0300736062",  # Franklin Natural Resources A(acc)USD, energy 54%
    "LU0612318385",  # BGF Natural Resources A2, energy 39%
    "LU1989770125",  # CPR Invest Global Resources A USD Acc, energy 32%
]

peer_ranks(ENERGY_ISIN, ENERGY_PEER_ISINS)


## Summary: which method to use

| | Cosine | Aitchison | Hellinger 60/40 |
|---|---|---|---|
| Clusters / not in a cluster | 47 / 47 | 41 / **85** | 46 / 53 |
| Energy neighbours (other 5) | all in top 6 | all in top 5 | all in top 5 |
| Defence neighbours (other 4) | 3 in top 20, one only at 10 | 3 in top 20, one at 7 | 3 in **top 4** |
| Short positions | used as reported | read as 0 | read as 0 |
| Country vs sector balance | accidental | fixed by the method | explicit, tunable |
| Easy to explain | yes | no (log-ratios, +0.005) | yes |

- **Default: Hellinger 60/40.** At least as good as Cosine on every test, a deliberate country/sector split, and a fixed 0..√2 distance range (useful for a similarity %).
- **Add a sector-leaning Hellinger 30/70** for "same theme" questions: all four other defence funds land in DNB's top 7, and energy stays all in the top 5.
- **Keep Cosine** as a familiar baseline.

Energy is an easy case (one dominant sector), so it doesn't separate the methods; defence does. Check a few more (Nordic small caps, tech, global) before switching the default.